# EDA — POS CASH Balance y Credit Card Balance

Análisis Exploratorio de Datos para las tablas de comportamiento mensual de créditos anteriores.

`pos_cash_balance`: Snapshots mensuales de créditos POS y préstamos en efectivo anteriores

`credit_card_balance`: Snapshots mensuales de tarjetas de crédito anteriores del cliente

Ambas tablas se unen a `application_train` vía `SK_ID_CURR` y a `previous_application` vía `SK_ID_PREV`.

In [3]:
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, NumericType

BUCKET_NAME  = 'hcdr'
TRUSTED_PATH = 's3://{}/trusted/'.format(BUCKET_NAME)
REFINED_PATH = 's3://{}/refined/eda/'.format(BUCKET_NAME)
TRUSTED_DB   = 'trusted_db'

print('Done')

Done


In [5]:
try:
    from awsglue.context import GlueContext
    from pyspark.context import SparkContext
    sc = SparkContext.getOrCreate()
    glueContext = GlueContext(sc)
    spark = glueContext.spark_session
    print('GlueContext OK')
except Exception:
    spark = (
        SparkSession.builder
        .appName('EDA_POS_CC')
        .config('spark.sql.adaptive.enabled', 'true')
        .config('spark.sql.shuffle.partitions', '120')
        .enableHiveSupport()
        .getOrCreate()
    )

spark.sparkContext.setLogLevel('WARN')
print('Spark version:', spark.version)

Exception in thread "main" java.lang.ExceptionInInitializerError
	at org.apache.spark.deploy.SparkSubmitArguments.$anonfun$loadEnvironmentArguments$5(SparkSubmitArguments.scala:171)
	at scala.Option.orElse(Option.scala:477)
	at org.apache.spark.deploy.SparkSubmitArguments.loadEnvironmentArguments(SparkSubmitArguments.scala:171)
	at org.apache.spark.deploy.SparkSubmitArguments.<init>(SparkSubmitArguments.scala:100)
	at org.apache.spark.deploy.SparkSubmit$$anon$2$$anon$3.<init>(SparkSubmit.scala:1137)
	at org.apache.spark.deploy.SparkSubmit$$anon$2.parseArguments(SparkSubmit.scala:1137)
	at org.apache.spark.deploy.SparkSubmit.doSubmit(SparkSubmit.scala:71)
	at org.apache.spark.deploy.SparkSubmit$$anon$2.doSubmit(SparkSubmit.scala:1168)
	at org.apache.spark.deploy.SparkSubmit$.main(SparkSubmit.scala:1177)
	at org.apache.spark.deploy.SparkSubmit.main(SparkSubmit.scala)
Caused by: java.net.UnknownHostException: jdk.internal.util.Exceptions$NonSocketInfo@545b995eOrfeo: Nombre o servicio desc

PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

In [ ]:
def sql(query, rows=30, truncate=False):
    result = spark.sql(query)
    result.show(rows, truncate=truncate)
    return result


def save_refined(df, name):
    out = REFINED_PATH + name + '/'
    df.coalesce(1).write.mode('overwrite') \
        .option('header', 'true').csv(out)
    print('Saved on refined/{}'.format(name))


def plot_bar(pdf, x, y, title, xlabel=None, ylabel=None, rotation=45, color='steelblue'):
    if pdf.empty:
        print('Empty df')
        return
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.bar(pdf[x].astype(str), pdf[y], color=color)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel(xlabel or x)
    ax.set_ylabel(ylabel or y)
    plt.xticks(rotation=rotation, ha='right' if rotation else 'center')
    plt.tight_layout()
    plt.show()


def plot_hbar(pdf, x, y, title, xlabel=None, ylabel=None, color='steelblue'):
    if pdf.empty:
        print('Empty df')
        return
    fig, ax = plt.subplots(figsize=(10, max(4, len(pdf) * 0.4)))
    ax.barh(pdf[y].astype(str), pdf[x], color=color)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel(xlabel or x)
    ax.set_ylabel(ylabel or y)
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()


def plot_dual_bar(pdf, x, y1, y2, title, label1='Total', label2='Con default',
                  rotation=45):
    if pdf.empty:
        return
    ind = range(len(pdf))
    fig, ax1 = plt.subplots(figsize=(12, 5))
    ax2 = ax1.twinx()
    ax1.bar([i - 0.2 for i in ind], pdf[y1], width=0.4, label=label1, color='steelblue', alpha=0.8)
    ax2.bar([i + 0.2 for i in ind], pdf[y2], width=0.4, label=label2, color='tomato', alpha=0.8)
    ax1.set_xticks(list(ind))
    ax1.set_xticklabels(pdf[x].astype(str), rotation=rotation, ha='right')
    ax1.set_ylabel(label1, color='steelblue')
    ax2.set_ylabel(label2, color='tomato')
    ax1.set_title(title, fontsize=13)
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')
    plt.tight_layout()
    plt.show()


def plot_line(pdf, x, y, title, xlabel=None, ylabel=None, color='steelblue'):
    if pdf.empty:
        return
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(pdf[x], pdf[y], marker='o', color=color)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel(xlabel or x)
    ax.set_ylabel(ylabel or y)
    plt.tight_layout()
    plt.show()

In [ ]:
def load_trusted(name):
    try:
        df = spark.table('{}.{}'.format(TRUSTED_DB, name))
        print('Glue Catalog OK:', name)
    except Exception:
        path = TRUSTED_PATH + name + '/'
        df = spark.read.parquet(path)
        print('S3 fallback OK:', name)
    df.createOrReplaceTempView(name)
    print('  {:,} rows x {} columns'.format(df.count(), len(df.columns)))
    return df


pos  = load_trusted('pos_cash_balance')
cc   = load_trusted('credit_card_balance')
app  = load_trusted('application_train')

app.select('sk_id_curr', 'target').createOrReplaceTempView('app_target')

S3 fallback OK: pos_cash_balance
  10,001,358 rows x 8 columns
S3 fallback OK: credit_card_balance
  3,840,312 rows x 23 columns
S3 fallback OK: application_train
  307,511 rows x 77 columns


---
### POS CASH Balance

Snapshots mensuales del comportamiento de créditos POS (punto de venta) y préstamos en efectivo anteriores. El diccionario completo de la base de datos esta en el archivo, que tenemos en la zona `raw` del bucket ***HomeCredit_columns_description.csv***

| Columna | Descripción |
|---|---|
| `sk_id_prev` | ID del crédito anterior |
| `sk_id_curr` | ID del préstamo actual (join con application) |
| `months_balance` | Mes relativo a la fecha de solicitud (-1 = más reciente) |
| `cnt_instalment` | Plazo del crédito anterior (puede cambiar) |
| `cnt_instalment_future` | Cuotas restantes por pagar |
| `name_contract_status` | Estado del contrato ese mes |
| `sk_dpd` | Días vencidos (Days Past Due) ese mes |
| `sk_dpd_def` | DPD con tolerancia (se ignoran montos pequeños) |

In [ ]:
pos.printSchema()

root
 |-- sk_id_prev: long (nullable = true)
 |-- sk_id_curr: long (nullable = true)
 |-- months_balance: long (nullable = true)
 |-- cnt_instalment: double (nullable = true)
 |-- cnt_instalment_future: double (nullable = true)
 |-- name_contract_status: string (nullable = true)
 |-- sk_dpd: long (nullable = true)
 |-- sk_dpd_def: long (nullable = true)


In [ ]:
pos_stats = sql("""
SELECT
    COUNT(*)                          AS total_registros,
    COUNT(DISTINCT sk_id_curr)        AS clientes_unicos,
    COUNT(DISTINCT sk_id_prev)        AS creditos_unicos,
    ROUND(COUNT(*) / COUNT(DISTINCT sk_id_prev), 1)
                                      AS snapshots_por_credito,
    MIN(months_balance)               AS mes_min,
    MAX(months_balance)               AS mes_max
FROM pos_cash_balance
""")

save_refined(pos_stats, 'pos_overview')

+---------------+---------------+---------------+---------------------+-------+-------+
|total_registros|clientes_unicos|creditos_unicos|snapshots_por_credito|mes_min|mes_max|
+---------------+---------------+---------------+---------------------+-------+-------+
|10001358       |337252         |936325         |10.7                 |-96    |-1     |
+---------------+---------------+---------------+---------------------+-------+-------+

Saved on refined/pos_overview


In [ ]:
pos_status = sql("""
SELECT
    name_contract_status,
    COUNT(*)                                      AS registros,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 2) AS pct
FROM pos_cash_balance
GROUP BY name_contract_status
ORDER BY registros DESC
""")

pos_status_pd = pos_status.toPandas()
plot_bar(
    pos_status_pd, x='name_contract_status', y='registros',
    title='POS CASH — Distribución de estado de contrato',
    xlabel='Estado', ylabel='Registros'
)

save_refined(pos_status, 'pos_contract_status')

+---------------------+---------+-----+
|name_contract_status |registros|pct  |
+---------------------+---------+-----+
|Active               |9151119  |91.50|
|Completed            |744883   |7.45 |
|Signed               |87260    |0.87 |
|Demand               |7065     |0.07 |
|Returned to the store|5461     |0.05 |
|Approved             |4917     |0.05 |
|Amortized debt       |636      |0.01 |
|Canceled             |15       |0.00 |
|XNA                  |2        |0.00 |
+---------------------+---------+-----+

Saved on refined/pos_contract_status


**Análisis de DPD**: `sk_dpd` y `sk_dpd_def` Un DPD > 0 es señal de riesgo.

`sk_dpd`: días vencidos reales.

`sk_dpd_def`: DPD con tolerancia (montos pequeños se ignoran).

In [ ]:
pos_dpd_stats = sql("""
SELECT
    'sk_dpd'     AS columna,
    MIN(sk_dpd)     AS min,
    MAX(sk_dpd)     AS max,
    ROUND(AVG(sk_dpd), 2) AS media,
    ROUND(PERCENTILE_APPROX(sk_dpd, 0.5), 2)  AS mediana,
    ROUND(PERCENTILE_APPROX(sk_dpd, 0.9), 2)  AS p90,
    ROUND(PERCENTILE_APPROX(sk_dpd, 0.99), 2) AS p99,
    SUM(CASE WHEN sk_dpd > 0 THEN 1 ELSE 0 END) AS registros_en_mora
FROM pos_cash_balance
UNION ALL
SELECT
    'sk_dpd_def',
    MIN(sk_dpd_def), MAX(sk_dpd_def),
    ROUND(AVG(sk_dpd_def), 2),
    ROUND(PERCENTILE_APPROX(sk_dpd_def, 0.5), 2),
    ROUND(PERCENTILE_APPROX(sk_dpd_def, 0.9), 2),
    ROUND(PERCENTILE_APPROX(sk_dpd_def, 0.99), 2),
    SUM(CASE WHEN sk_dpd_def > 0 THEN 1 ELSE 0 END)
FROM pos_cash_balance
""")

save_refined(pos_dpd_stats, 'pos_dpd_stats')

+----------+---+----+-----+-------+---+---+-----------------+
|columna   |min|max |media|mediana|p90|p99|registros_en_mora|
+----------+---+----+-----+-------+---+---+-----------------+
|sk_dpd    |0  |4231|11.61|0      |0  |233|295227           |
|sk_dpd_def|0  |3595|0.65 |0      |0  |1  |113969           |
+----------+---+----+-----+-------+---+---+-----------------+

Saved on refined/pos_dpd_stats


In [ ]:
pos_dpd_dist = sql("""
SELECT
    CASE
        WHEN sk_dpd = 0              THEN '0 (sin mora)'
        WHEN sk_dpd BETWEEN 1  AND 30  THEN '1-30'
        WHEN sk_dpd BETWEEN 31 AND 60  THEN '31-60'
        WHEN sk_dpd BETWEEN 61 AND 90  THEN '61-90'
        ELSE '90+'
    END AS bucket_dpd,
    COUNT(*) AS registros,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 2) AS pct
FROM pos_cash_balance
GROUP BY 1
ORDER BY
    CASE bucket_dpd
        WHEN '0 (sin mora)' THEN 0
        WHEN '1-30'  THEN 1
        WHEN '31-60' THEN 2
        WHEN '61-90' THEN 3
        ELSE 4
    END
""")

pos_dpd_dist_pd = pos_dpd_dist.toPandas()
plot_bar(
    pos_dpd_dist_pd, x='bucket_dpd', y='registros',
    title='POS CASH — Distribución de DPD (sk_dpd)',
    xlabel='Rango de días vencidos', ylabel='Registros', rotation=0
)

save_refined(pos_dpd_dist, 'pos_dpd_distribution')

+------------+---------+-----+
|bucket_dpd  |registros|pct  |
+------------+---------+-----+
|0 (sin mora)|9706131  |97.05|
|1-30        |163169   |1.63 |
|31-60       |7944     |0.08 |
|61-90       |4996     |0.05 |
|90+         |119118   |1.19 |
+------------+---------+-----+

Saved on refined/pos_dpd_distribution


In [ ]:
# cnt_instalment es el plazo total del crédito y cnt_instalment_future son las cuotas pendientes. La diferencia muestra cuánto lleva pagado el cliente.

pos_inst = sql("""
SELECT
    ROUND(AVG(cnt_instalment), 2)        AS plazo_promedio,
    ROUND(AVG(cnt_instalment_future), 2) AS cuotas_restantes_promedio,
    ROUND(AVG(cnt_instalment - cnt_instalment_future), 2) AS cuotas_pagadas_promedio,
    ROUND(
        AVG(
            CASE WHEN cnt_instalment > 0
                 THEN (cnt_instalment - cnt_instalment_future) / cnt_instalment
                 ELSE NULL END
        ) * 100, 2
    ) AS pct_completado_promedio
FROM pos_cash_balance
WHERE cnt_instalment IS NOT NULL
  AND cnt_instalment_future IS NOT NULL
""")

save_refined(pos_inst, 'pos_installment_summary')

+--------------+-------------------------+-----------------------+-----------------------+
|plazo_promedio|cuotas_restantes_promedio|cuotas_pagadas_promedio|pct_completado_promedio|
+--------------+-------------------------+-----------------------+-----------------------+
|17.09         |10.48                    |6.61                   |45.3                   |
+--------------+-------------------------+-----------------------+-----------------------+

Saved on refined/pos_installment_summary


In [ ]:
pos_plazo = sql("""
SELECT
    CAST(cnt_instalment AS INT) AS plazo,
    COUNT(*) AS registros
FROM pos_cash_balance
WHERE cnt_instalment IS NOT NULL
GROUP BY 1
ORDER BY registros DESC
LIMIT 20
""")

pos_plazo_pd = pos_plazo.orderBy('plazo').toPandas()
plot_bar(
    pos_plazo_pd, x='plazo', y='registros',
    title='POS CASH — Top 20 plazos de crédito (cnt_instalment)',
    xlabel='Plazo (meses)', ylabel='Registros'
)

save_refined(pos_plazo, 'pos_plazo_distribution')

+-----+---------+
|plazo|registros|
+-----+---------+
|12   |2496845  |
|24   |1517472  |
|10   |1243449  |
|6    |1065500  |
|18   |727394   |
|36   |584574   |
|8    |303751   |
|48   |278513   |
|4    |238223   |
|30   |211920   |
|60   |189067   |
|11   |151835   |
|9    |148355   |
|5    |136840   |
|7    |106714   |
|14   |90538    |
|42   |87935    |
|16   |66426    |
|3    |43081    |
|20   |33459    |
+-----+---------+

Saved on refined/pos_plazo_distribution


In [ ]:
# ¿En que momento del ciclo de vida del credito se concentra la mora?

pos_temporal = sql("""
SELECT
    months_balance,
    COUNT(*)                                       AS registros,
    SUM(CASE WHEN sk_dpd > 0 THEN 1 ELSE 0 END)   AS en_mora,
    ROUND(
        100.0 * SUM(CASE WHEN sk_dpd > 0 THEN 1 ELSE 0 END) / COUNT(*),
    2) AS tasa_mora_pct,
    ROUND(AVG(sk_dpd), 3)                          AS dpd_promedio
FROM pos_cash_balance
GROUP BY months_balance
ORDER BY months_balance
""", rows=10)

pos_temporal_pd = pos_temporal.toPandas()

fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()
ax1.bar(pos_temporal_pd['months_balance'], pos_temporal_pd['registros'],
        color='steelblue', alpha=0.5, label='Registros')
ax2.plot(pos_temporal_pd['months_balance'], pos_temporal_pd['tasa_mora_pct'],
         color='tomato', linewidth=2, label='% Mora')
ax1.set_xlabel('Mes (relativo a solicitud)')
ax1.set_ylabel('Registros', color='steelblue')
ax2.set_ylabel('Tasa de mora (%)', color='tomato')
ax1.set_title('POS CASH — Volumen y tasa de mora por mes')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2)
plt.tight_layout()
plt.show()

save_refined(pos_temporal, 'pos_mora_temporal')

+--------------+---------+-------+-------------+------------+
|months_balance|registros|en_mora|tasa_mora_pct|dpd_promedio|
+--------------+---------+-------+-------------+------------+
|-96           |36448    |2080   |5.71         |6.534       |
|-95           |38514    |2137   |5.55         |6.778       |
|-94           |39900    |2227   |5.58         |7.083       |
|-93           |41025    |2216   |5.40         |7.447       |
|-92           |42283    |2319   |5.48         |7.779       |
|-91           |43652    |2396   |5.49         |8.088       |
|-90           |45295    |2444   |5.40         |8.377       |
|-89           |47763    |2463   |5.16         |8.485       |
|-88           |49950    |2606   |5.22         |8.667       |
|-87           |51805    |2740   |5.29         |8.949       |
+--------------+---------+-------+-------------+------------+
only showing top 10 rows

Saved on refined/pos_mora_temporal


In [ ]:
# Relación con el target ¿diferencia entre clientes en default y no default?

pos_agg = sql("""
SELECT
    p.sk_id_curr,
    t.target,
    COUNT(*)                                        AS total_snapshots,
    COUNT(DISTINCT p.sk_id_prev)                    AS creditos_pos,
    SUM(CASE WHEN p.sk_dpd > 0 THEN 1 ELSE 0 END)  AS meses_en_mora,
    MAX(p.sk_dpd)                                   AS max_dpd,
    ROUND(AVG(p.sk_dpd), 2)                         AS avg_dpd,
    MAX(p.cnt_instalment)                           AS max_plazo,
    -- Status más frecuente (moda aproximada)
    FIRST(p.name_contract_status)                   AS status_frecuente
FROM pos_cash_balance p
JOIN app_target t ON p.sk_id_curr = t.sk_id_curr
WHERE t.target IN (0, 1)
GROUP BY p.sk_id_curr, t.target
""")

pos_agg.createOrReplaceTempView('pos_agg_target')

pos_default_summary = sql("""
SELECT
    target,
    COUNT(*)                           AS clientes,
    ROUND(AVG(total_snapshots), 1)     AS avg_snapshots,
    ROUND(AVG(creditos_pos), 2)        AS avg_creditos_pos,
    ROUND(AVG(meses_en_mora), 2)       AS avg_meses_mora,
    ROUND(AVG(max_dpd), 2)             AS avg_max_dpd,
    ROUND(AVG(avg_dpd), 4)             AS avg_dpd_promedio
FROM pos_agg_target
GROUP BY target
ORDER BY target
""")

save_refined(pos_default_summary, 'pos_default_comparison')

+----------+------+---------------+------------+-------------+-------+-------+---------+----------------+
|sk_id_curr|target|total_snapshots|creditos_pos|meses_en_mora|max_dpd|avg_dpd|max_plazo|status_frecuente|
+----------+------+---------------+------------+-------------+-------+-------+---------+----------------+
|100002    |1     |19             |1           |0            |0      |0.0    |24.0     |Active          |
|100051    |0     |18             |1           |0            |0      |0.0    |24.0     |Active          |
|100075    |0     |18             |3           |0            |0      |0.0    |6.0      |Active          |
|100112    |1     |21             |3           |0            |0      |0.0    |10.0     |Active          |
|100127    |0     |19             |2           |2            |22     |2.11   |12.0     |Active          |
|100145    |0     |62             |5           |0            |0      |0.0    |48.0     |Active          |
|100148    |0     |29             |2          

In [ ]:
pos_dpd_target = sql("""
SELECT
    target,
    CASE
        WHEN max_dpd = 0             THEN '0'
        WHEN max_dpd BETWEEN 1 AND 30  THEN '1-30'
        WHEN max_dpd BETWEEN 31 AND 60 THEN '31-60'
        WHEN max_dpd BETWEEN 61 AND 90 THEN '61-90'
        ELSE '90+'
    END AS bucket_max_dpd,
    COUNT(*) AS clientes
FROM pos_agg_target
GROUP BY 1, 2
ORDER BY target, 2
""")

pos_dpd_target_pd = pos_dpd_target.toPandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)
order = ['0', '1-30', '31-60', '61-90', '90+']
for ax, tgt, label in zip(axes, [0, 1], ['Sin default (target=0)', 'Default (target=1)']):
    sub = pos_dpd_target_pd[pos_dpd_target_pd['target'] == tgt]
    sub = sub.set_index('bucket_max_dpd').reindex(order).reset_index()
    ax.bar(sub['bucket_max_dpd'], sub['clientes'].fillna(0),
           color='steelblue' if tgt == 0 else 'tomato')
    ax.set_title('POS CASH — Max DPD | {}'.format(label))
    ax.set_xlabel('Rango DPD máximo')
    ax.set_ylabel('Clientes')
plt.tight_layout()
plt.show()

save_refined(pos_dpd_target, 'pos_dpd_by_target')

+------+--------------+--------+
|target|bucket_max_dpd|clientes|
+------+--------------+--------+
|0     |0             |216639  |
|0     |1-30          |42694   |
|0     |31-60         |1849    |
|0     |61-90         |394     |
|0     |90+           |4247    |
|1     |0             |18122   |
|1     |1-30          |4815    |
|1     |31-60         |230     |
|1     |61-90         |59      |
|1     |90+           |395     |
+------+--------------+--------+

Saved on refined/pos_dpd_by_target


In [ ]:
pos_last_status = sql("""
WITH latest AS (
    SELECT
        sk_id_curr,
        sk_id_prev,
        name_contract_status,
        ROW_NUMBER() OVER (PARTITION BY sk_id_curr, sk_id_prev ORDER BY months_balance DESC) AS rn
    FROM pos_cash_balance
)
SELECT
    l.name_contract_status,
    t.target,
    COUNT(DISTINCT l.sk_id_curr)        AS clientes,
    COUNT(*)                            AS creditos
FROM latest l
JOIN app_target t ON l.sk_id_curr = t.sk_id_curr
WHERE l.rn = 1
  AND t.target IN (0, 1)
GROUP BY l.name_contract_status, t.target
ORDER BY l.name_contract_status, t.target
""")

pos_last_status_pd = pos_last_status.toPandas()

pivot = pos_last_status_pd.pivot(index='name_contract_status', columns='target', values='creditos').fillna(0)
pivot.columns = ['Sin default', 'Default']
pivot.plot(kind='bar', figsize=(12, 5), color=['steelblue', 'tomato'])
plt.title('POS CASH — Estado del contrato más reciente vs Target')
plt.xlabel('Estado')
plt.ylabel('Créditos')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

save_refined(pos_last_status, 'pos_last_status_by_target')

+---------------------+------+--------+--------+
|name_contract_status |target|clientes|creditos|
+---------------------+------+--------+--------+
|Active               |0     |144088  |188749  |
|Active               |1     |11740   |14994   |
|Amortized debt       |0     |6       |6       |
|Amortized debt       |1     |5       |5       |
|Approved             |0     |51      |51      |
|Approved             |1     |3       |3       |
|Canceled             |0     |2       |2       |
|Completed            |0     |236698  |551221  |
|Completed            |1     |20496   |43855   |
|Demand               |0     |70      |74      |
|Demand               |1     |16      |18      |
|Returned to the store|0     |231     |231     |
|Returned to the store|1     |44      |44      |
|Signed               |0     |989     |990     |
|Signed               |1     |94      |94      |
+---------------------+------+--------+--------+

Saved on refined/pos_last_status_by_target


---
### Credit Card Balance

Snapshots mensuales del comportamiento de tarjetas de crédito anteriores del cliente.

| Columna | Descripción |
|---|---|
| `amt_balance` | Saldo de la tarjeta ese mes |
| `amt_credit_limit_actual` | Límite de crédito vigente |
| `amt_drawings_*` | Montos usados (ATM / compras POS / otros) |
| `amt_payment_current` | Pago realizado por el cliente ese mes |
| `amt_inst_min_regularity` | Pago mínimo requerido |
| `cnt_instalment_mature_cum` | Cuotas acumuladas pagadas |
| `sk_dpd` | Días vencidos ese mes |
| `name_contract_status` | Estado del contrato ese mes |

In [ ]:
cc.printSchema()

cc_stats = sql("""
SELECT
    COUNT(*)                          AS total_registros,
    COUNT(DISTINCT sk_id_curr)        AS clientes_unicos,
    COUNT(DISTINCT sk_id_prev)        AS tarjetas_unicas,
    ROUND(COUNT(*) / COUNT(DISTINCT sk_id_prev), 1)
                                      AS snapshots_por_tarjeta,
    MIN(months_balance)               AS mes_min,
    MAX(months_balance)               AS mes_max
FROM credit_card_balance
""")

save_refined(cc_stats, 'cc_overview')

root
 |-- sk_id_prev: long (nullable = true)
 |-- sk_id_curr: long (nullable = true)
 |-- months_balance: long (nullable = true)
 |-- amt_balance: double (nullable = true)
 |-- amt_credit_limit_actual: long (nullable = true)
 |-- amt_drawings_atm_current: double (nullable = true)
 |-- amt_drawings_current: double (nullable = true)
 |-- amt_drawings_other_current: double (nullable = true)
 |-- amt_drawings_pos_current: double (nullable = true)
 |-- amt_inst_min_regularity: double (nullable = true)
 |-- amt_payment_current: double (nullable = true)
 |-- amt_payment_total_current: double (nullable = true)
 |-- amt_receivable_principal: double (nullable = true)
 |-- amt_recivable: double (nullable = true)
 |-- amt_total_receivable: double (nullable = true)
 |-- cnt_drawings_atm_current: double (nullable = true)
 |-- cnt_drawings_current: long (nullable = true)
 |-- cnt_drawings_other_current: double (nullable = true)
 |-- cnt_drawings_pos_current: double (nullable = true)
 |-- cnt_instalme

In [ ]:
# amt_balance / amt_credit_limit_actual. Lo consideramos relevante ya que nos puede decir que tanto margen de acción tiene una persona
#    los que tengan un porcentaje muy alto es probable que incurran en incumplimientos más facilmente 

cc_balance = sql("""
SELECT
    ROUND(MIN(amt_balance), 2)            AS saldo_min,
    ROUND(MAX(amt_balance), 2)            AS saldo_max,
    ROUND(AVG(amt_balance), 2)            AS saldo_promedio,
    ROUND(PERCENTILE_APPROX(amt_balance, 0.5),  2) AS saldo_mediana,
    ROUND(PERCENTILE_APPROX(amt_balance, 0.9),  2) AS saldo_p90,
    ROUND(AVG(amt_credit_limit_actual), 2)         AS limite_promedio,
    ROUND(
        AVG(
            CASE WHEN amt_credit_limit_actual > 0
                 THEN amt_balance / amt_credit_limit_actual
                 ELSE NULL END
        ) * 100, 2
    ) AS utilizacion_pct_promedio
FROM credit_card_balance
WHERE amt_balance IS NOT NULL
  AND amt_credit_limit_actual IS NOT NULL
""")

save_refined(cc_balance, 'cc_balance_stats')

+----------+----------+--------------+-------------+---------+---------------+------------------------+
|saldo_min |saldo_max |saldo_promedio|saldo_mediana|saldo_p90|limite_promedio|utilizacion_pct_promedio|
+----------+----------+--------------+-------------+---------+---------------+------------------------+
|-420250.19|1505902.19|58300.16      |0.0          |180004.68|153807.96      |37.47                   |
+----------+----------+--------------+-------------+---------+---------------+------------------------+

Saved on refined/cc_balance_stats


In [ ]:
cc_util = sql("""
SELECT
    CASE
        WHEN amt_credit_limit_actual <= 0 THEN 'Sin límite'
        WHEN amt_balance / amt_credit_limit_actual <= 0.25 THEN '0-25%'
        WHEN amt_balance / amt_credit_limit_actual <= 0.50 THEN '25-50%'
        WHEN amt_balance / amt_credit_limit_actual <= 0.75 THEN '50-75%'
        WHEN amt_balance / amt_credit_limit_actual <= 1.00 THEN '75-100%'
        ELSE '>100%'
    END AS bucket_utilizacion,
    COUNT(*) AS registros,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 2) AS pct
FROM credit_card_balance
WHERE amt_balance IS NOT NULL
  AND amt_credit_limit_actual IS NOT NULL
GROUP BY 1
ORDER BY
    CASE bucket_utilizacion
        WHEN '0-25%'      THEN 0
        WHEN '25-50%'     THEN 1
        WHEN '50-75%'     THEN 2
        WHEN '75-100%'    THEN 3
        WHEN '>100%'      THEN 4
        ELSE 5
    END
""")

cc_util_pd = cc_util.toPandas()
plot_bar(
    cc_util_pd, x='bucket_utilizacion', y='registros',
    title='Credit Card — Distribución de utilización del crédito',
    xlabel='Rango de utilización', ylabel='Registros', rotation=0
)

save_refined(cc_util, 'cc_utilizacion_distribution')

+------------------+---------+-----+
|bucket_utilizacion|registros|pct  |
+------------------+---------+-----+
|0-25%             |1699097  |44.24|
|25-50%            |183660   |4.78 |
|50-75%            |247974   |6.46 |
|75-100%           |621396   |16.18|
|>100%             |334362   |8.71 |
|Sin límite        |753823   |19.63|
+------------------+---------+-----+

Saved on refined/cc_utilizacion_distribution


In [ ]:
cc_drawings = sql("""
SELECT
    ROUND(AVG(amt_drawings_current), 2)        AS avg_retiro_total,
    ROUND(AVG(amt_drawings_atm_current), 2)    AS avg_retiro_atm,
    ROUND(AVG(amt_drawings_pos_current), 2)    AS avg_retiro_pos,
    ROUND(AVG(amt_drawings_other_current), 2)  AS avg_retiro_otro,
    ROUND(AVG(amt_payment_current), 2)         AS avg_pago,
    ROUND(AVG(amt_inst_min_regularity), 2)     AS avg_pago_minimo,
    -- % de meses donde el pago no alcanza el mínimo
    ROUND(
        100.0 * SUM(
            CASE WHEN amt_inst_min_regularity IS NOT NULL
                  AND amt_payment_current IS NOT NULL
                  AND amt_payment_current < amt_inst_min_regularity
                 THEN 1 ELSE 0 END
        ) / COUNT(*), 2
    ) AS pct_pago_menor_minimo
FROM credit_card_balance
""")

save_refined(cc_drawings, 'cc_payments_summary')

+----------------+--------------+--------------+---------------+--------+---------------+---------------------+
|avg_retiro_total|avg_retiro_atm|avg_retiro_pos|avg_retiro_otro|avg_pago|avg_pago_minimo|pct_pago_menor_minimo|
+----------------+--------------+--------------+---------------+--------+---------------+---------------------+
|7433.39         |5961.32       |2968.8        |288.17         |10280.54|3540.2         |3.23                 |
+----------------+--------------+--------------+---------------+--------+---------------+---------------------+

Saved on refined/cc_payments_summary


In [ ]:
cc_draw_mix = sql("""
SELECT
    'ATM'   AS tipo,
    ROUND(SUM(amt_drawings_atm_current), 0)   AS monto_total
FROM credit_card_balance WHERE amt_drawings_atm_current > 0
UNION ALL
SELECT 'POS',
    ROUND(SUM(amt_drawings_pos_current), 0)
FROM credit_card_balance WHERE amt_drawings_pos_current > 0
UNION ALL
SELECT 'Otros',
    ROUND(SUM(amt_drawings_other_current), 0)
FROM credit_card_balance WHERE amt_drawings_other_current > 0
""")

cc_draw_mix_pd = cc_draw_mix.toPandas()
fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(
    cc_draw_mix_pd['monto_total'],
    labels=cc_draw_mix_pd['tipo'],
    autopct='%1.1f%%',
    colors=['steelblue', 'seagreen', 'goldenrod']
)
ax.set_title('Credit Card — Composición del monto total retirado')
plt.tight_layout()
plt.show()

save_refined(cc_draw_mix, 'cc_drawings_mix')

+-----+---------------+
|tipo |monto_total    |
+-----+---------------+
|ATM  |1.8423457345E10|
|POS  |9.175079507E9  |
|Otros|8.90586942E8   |
+-----+---------------+

Saved on refined/cc_drawings_mix


In [ ]:
cc_dpd_stats = sql("""
SELECT
    name_contract_status,
    COUNT(*)                                      AS registros,
    ROUND(AVG(sk_dpd), 3)                         AS dpd_promedio,
    MAX(sk_dpd)                                   AS dpd_max,
    SUM(CASE WHEN sk_dpd > 0 THEN 1 ELSE 0 END)  AS meses_en_mora,
    ROUND(
        100.0 * SUM(CASE WHEN sk_dpd > 0 THEN 1 ELSE 0 END) / COUNT(*),
    2) AS pct_mora
FROM credit_card_balance
GROUP BY name_contract_status
ORDER BY dpd_promedio DESC
""")

cc_dpd_stats_pd = cc_dpd_stats.toPandas()
plot_bar(
    cc_dpd_stats_pd, x='name_contract_status', y='pct_mora',
    title='Credit Card — % de meses en mora por estado de contrato',
    xlabel='Estado', ylabel='% Meses en mora'
)

save_refined(cc_dpd_stats, 'cc_dpd_by_status')

In [ ]:
cc_temporal = sql("""
SELECT
    months_balance,
    COUNT(*)                                       AS registros,
    ROUND(AVG(amt_balance), 2)                     AS saldo_promedio,
    ROUND(
        AVG(
            CASE WHEN amt_credit_limit_actual > 0
                 THEN amt_balance / amt_credit_limit_actual
                 ELSE NULL END
        ) * 100, 2
    ) AS utilizacion_pct,
    ROUND(AVG(sk_dpd), 3)                          AS dpd_promedio,
    SUM(CASE WHEN sk_dpd > 0 THEN 1 ELSE 0 END)   AS en_mora
FROM credit_card_balance
GROUP BY months_balance
ORDER BY months_balance
""", rows=10)

cc_temporal_pd = cc_temporal.toPandas()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax1.plot(cc_temporal_pd['months_balance'], cc_temporal_pd['saldo_promedio'],
         color='steelblue', linewidth=2)
ax1.set_title('Credit Card — Saldo promedio por mes')
ax1.set_ylabel('Saldo promedio')

ax2.plot(cc_temporal_pd['months_balance'], cc_temporal_pd['utilizacion_pct'],
         color='darkorange', linewidth=2)
ax2.set_title('Credit Card — Utilización promedio por mes')
ax2.set_ylabel('Utilización (%)')
ax2.set_xlabel('Mes (relativo a solicitud)')

plt.tight_layout()
plt.show()

save_refined(cc_temporal, 'cc_temporal_trend')

+--------------+---------+--------------+---------------+------------+-------+
|months_balance|registros|saldo_promedio|utilizacion_pct|dpd_promedio|en_mora|
+--------------+---------+--------------+---------------+------------+-------+
|-96           |11722    |63434.47      |53.01          |0.393       |870    |
|-95           |12521    |63763.45      |53.12          |0.384       |920    |
|-94           |13397    |64443.61      |53.22          |0.435       |977    |
|-93           |14197    |65817.71      |53.93          |0.439       |1095   |
|-92           |14911    |66583.99      |54.15          |0.419       |1136   |
|-91           |15656    |67027.36      |54.3           |0.366       |1116   |
|-90           |16385    |66975.08      |54.1           |0.337       |1109   |
|-89           |17088    |67469.92      |54.31          |0.352       |1087   |
|-88           |17806    |67253.72      |53.92          |0.38        |1101   |
|-87           |18465    |67428.86      |53.91      

In [ ]:
cc_agg = sql("""
SELECT
    c.sk_id_curr,
    t.target,
    COUNT(*)                                              AS total_snapshots,
    COUNT(DISTINCT c.sk_id_prev)                          AS tarjetas,
    ROUND(AVG(c.amt_balance), 2)                          AS saldo_promedio,
    ROUND(MAX(c.amt_balance), 2)                          AS saldo_max,
    ROUND(
        AVG(
            CASE WHEN c.amt_credit_limit_actual > 0
                 THEN c.amt_balance / c.amt_credit_limit_actual
                 ELSE NULL END
        ) * 100, 2
    )                                                     AS utilizacion_pct,
    SUM(CASE WHEN c.sk_dpd > 0 THEN 1 ELSE 0 END)       AS meses_mora,
    MAX(c.sk_dpd)                                         AS max_dpd,
    ROUND(
        SUM(CASE WHEN c.amt_payment_current < c.amt_inst_min_regularity
                      AND c.amt_inst_min_regularity IS NOT NULL
                 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2
    )                                                     AS pct_pago_bajo_minimo
FROM credit_card_balance c
JOIN app_target t ON c.sk_id_curr = t.sk_id_curr
WHERE t.target IN (0, 1)
GROUP BY c.sk_id_curr, t.target
""")

cc_agg.createOrReplaceTempView('cc_agg_target')

cc_default_summary = sql("""
SELECT
    target,
    COUNT(*)                                AS clientes,
    ROUND(AVG(tarjetas), 2)                 AS avg_tarjetas,
    ROUND(AVG(saldo_promedio), 2)           AS avg_saldo,
    ROUND(AVG(utilizacion_pct), 2)          AS avg_utilizacion_pct,
    ROUND(AVG(meses_mora), 2)               AS avg_meses_mora,
    ROUND(AVG(max_dpd), 2)                  AS avg_max_dpd,
    ROUND(AVG(pct_pago_bajo_minimo), 2)     AS avg_pct_pago_bajo_minimo
FROM cc_agg_target
GROUP BY target
ORDER BY target
""")

save_refined(cc_default_summary, 'cc_default_comparison')

+----------+------+---------------+--------+--------------+---------+---------------+----------+-------+--------------------+
|sk_id_curr|target|total_snapshots|tarjetas|saldo_promedio|saldo_max|utilizacion_pct|meses_mora|max_dpd|pct_pago_bajo_minimo|
+----------+------+---------------+--------+--------------+---------+---------------+----------+-------+--------------------+
|404942    |0     |22             |1       |202679.69     |239706.41|90.08          |0         |0      |9.09                |
|371622    |1     |96             |1       |36500.98      |125166.24|32.45          |29        |336    |15.63               |
|170103    |0     |34             |1       |330357.38     |676288.04|52.9           |0         |0      |0.00                |
|194016    |0     |29             |1       |211293.26     |566399.66|36.38          |0         |0      |0.00                |
|220399    |0     |28             |1       |143316.08     |325208.12|38.08          |0         |0      |3.57          

In [ ]:
cc_util_target = sql("""
SELECT
    target,
    CASE
        WHEN utilizacion_pct IS NULL      THEN 'N/A'
        WHEN utilizacion_pct <= 25        THEN '0-25%'
        WHEN utilizacion_pct <= 50        THEN '25-50%'
        WHEN utilizacion_pct <= 75        THEN '50-75%'
        WHEN utilizacion_pct <= 100       THEN '75-100%'
        ELSE '>100%'
    END AS bucket_util,
    COUNT(*) AS clientes
FROM cc_agg_target
GROUP BY 1, 2
ORDER BY target, 2
""")

cc_util_target_pd = cc_util_target.toPandas()
order = ['0-25%', '25-50%', '50-75%', '75-100%', '>100%', 'N/A']

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)
for ax, tgt, label, color in zip(
    axes, [0, 1],
    ['Sin default (target=0)', 'Default (target=1)'],
    ['steelblue', 'tomato']
):
    sub = cc_util_target_pd[cc_util_target_pd['target'] == tgt]
    sub = sub.set_index('bucket_util').reindex(order).reset_index()
    ax.bar(sub['bucket_util'], sub['clientes'].fillna(0), color=color)
    ax.set_title('CC Utilización | {}'.format(label))
    ax.set_xlabel('Bucket utilización')
    ax.set_ylabel('Clientes')
plt.tight_layout()
plt.show()

save_refined(cc_util_target, 'cc_utilizacion_by_target')

+------+-----------+--------+
|target|bucket_util|clientes|
+------+-----------+--------+
|0     |0-25%      |40334   |
|0     |25-50%     |14972   |
|0     |50-75%     |12315   |
|0     |75-100%    |10238   |
|0     |>100%      |746     |
|0     |N/A        |766     |
|1     |0-25%      |2442    |
|1     |25-50%     |1321    |
|1     |50-75%     |1549    |
|1     |75-100%    |1863    |
|1     |>100%      |256     |
|1     |N/A        |103     |
+------+-----------+--------+

Saved on refined/cc_utilizacion_by_target


In [ ]:
cross = sql("""
WITH pos_clients AS (
    SELECT DISTINCT sk_id_curr FROM pos_cash_balance
),
cc_clients AS (
    SELECT DISTINCT sk_id_curr FROM credit_card_balance
)
SELECT
    t.target,
    CASE
        WHEN p.sk_id_curr IS NOT NULL AND c.sk_id_curr IS NOT NULL THEN 'Ambos'
        WHEN p.sk_id_curr IS NOT NULL                              THEN 'Solo POS'
        WHEN c.sk_id_curr IS NOT NULL                              THEN 'Solo CC'
        ELSE 'Ninguno'
    END AS tipo_historial,
    COUNT(DISTINCT t.sk_id_curr) AS clientes
FROM app_target t
LEFT JOIN pos_clients p ON t.sk_id_curr = p.sk_id_curr
LEFT JOIN cc_clients  c ON t.sk_id_curr = c.sk_id_curr
WHERE t.target IN (0, 1)
GROUP BY t.target, tipo_historial
ORDER BY tipo_historial, t.target
""")

cross_pd = cross.toPandas()

pivot = cross_pd.pivot(index='tipo_historial', columns='target', values='clientes').fillna(0)
pivot.columns = ['Sin default', 'Default']
pivot['default_rate'] = pivot['Default'] / (pivot['Sin default'] + pivot['Default']) * 100
print(pivot.round(2))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

pivot[['Sin default', 'Default']].plot(kind='bar', ax=ax1,
    color=['steelblue', 'tomato'])
ax1.set_title('Clientes por tipo de historial y target')
ax1.set_xlabel('Tipo de historial')
ax1.set_ylabel('Clientes')
ax1.tick_params(axis='x', rotation=0)

ax2.bar(pivot.index, pivot['default_rate'], color='darkorange')
ax2.set_title('Tasa de default por tipo de historial')
ax2.set_xlabel('Tipo de historial')
ax2.set_ylabel('Tasa de default (%)')
ax2.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

save_refined(cross, 'cross_historial_target')

+------+--------------+--------+
|target|tipo_historial|clientes|
+------+--------------+--------+
|0     |Ambos         |76863   |
|1     |Ambos         |7213    |
|0     |Ninguno       |14355   |
|1     |Ninguno       |883     |
|0     |Solo CC       |2508    |
|1     |Solo CC       |321     |
|0     |Solo POS      |188960  |
|1     |Solo POS      |16408   |
+------+--------------+--------+

                Sin default  Default  default_rate
tipo_historial                                    
Ambos                 76863     7213          8.58
Ninguno               14355      883          5.79
Solo CC                2508      321         11.35
Solo POS             188960    16408          7.99
Saved on refined/cross_historial_target


### Conclusiones EDA — POS CASH Balance y Credit Card Balance

Segun nuestro EDA estas tablas son buenas candidatas para ser utilizadas, en particular ***credit_card_balance*** fue la que demostro tener más potencial de predicción, ***pos_cash_balance*** tiene un impacto bastante más moderado.

**Credit Card Balance**: La tabla tiene más de 3 millones de snapshots de 103k clientes con un promedio de 36.8 meses de historia por tarjeta, suficiente para calcular agregaciones confiables por cliente.

El hallazgo más importante del EDA es la utilización del crédito (amt_balance / amt_credit_limit_actual): los clientes en default tienen una utilización promedio de 47% vs 31% en clientes sin default, una diferencia de 15. Este es el predictor más fuerte de esta tabla.

El porcentaje de meses donde el pago fue menor al mínimo requerido también separa las clases, 5.3% en clientes con default vs 3.6% sin default. Esto probablemente es una señal de estres económico en los clientes.

Los retiros ATM acumulan el doble en monto que los retiros POS en el historial completo, lo que sugiere dependencia al efectivo — comportamiento asociado a mayor riesgo.

**Features a construir:**
- cc_utilizacion_avg   = AVG(amt_balance / amt_credit_limit_actual) por cliente
- cc_utilizacion_max   = MAX(amt_balance / amt_credit_limit_actual) por cliente
- cc_pct_pago_bajo_min = % meses con amt_payment_current < amt_inst_min_regularity
- cc_dpd_max           = MAX(sk_dpd) por cliente
- cc_meses_mora        = SUM(sk_dpd > 0) por cliente

**Pos Cash Balance**: La tabla tiene 10 millones de snapshots de 337k clientes. Los clientes en default tienen avg_max_dpd de 18.15 vs 15.48 y avg_meses_mora de 1.09 vs 0.88. Las diferencias existen pero son pequeñas, probablemente porque la mayoría de defaults en el target actual son primeros incumplimientos sin historial previo de mora en POS.

El feature más informativo no es el DPD sino el avance del crédito `pct_completado = (cnt_instalment - cnt_instalment_future) / cnt_instalment`, con un promedio global de 45.3% y variabilidad suficiente como para considerarla en nuestro modelo.

Para DPD se debe usar sk_dpd_def en lugar de sk_dpd. El EDA mostró que 295k registros tienen mora bruta pero solo 114k tienen mora con tolerancia,la diferencia corresponde a montos pequeños que el banco ignora. sk_dpd_def es la señal más limpia de incumplimiento real.

**Features a construir:**
- pos_dpd_max        = MAX(sk_dpd_def) por cliente
- pos_meses_mora     = SUM(sk_dpd_def > 0) por cliente
- pos_pct_completado = AVG((cnt_instalment - cnt_instalment_future) / cnt_instalment)
- pos_creditos       = COUNT(DISTINCT sk_id_prev) por cliente

El análisis cruzado reveló que los clientes con solo historial de tarjeta de crédito tienen la tasa de default más alta (11.35%), por encima de quienes tienen ambos productos (8.58%) o solo POS (7.99%). Una variable categórica de tipo de historial puede aportar señal incremental sin costo adicional.